In [ ]:
!pip install -U langchain langchain-openai langchain-community langchain-classic faiss-cpu tiktoken


In [ ]:
import os
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

loader = TextLoader("faq.txt") #Ensure the file exists
documents = loader.load() #Load the documents from the file
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

#Set the Azure OpenAi API key
os.environ["AZURE_OPENAI_API_KEY"] = "YOUR-OPENAI-KEY" #Replace with your Azure OpenAI API key

#Generate embeddings for the documents
embeddings = AzureOpenAIEmbeddings(
 azure_endpoint="https://openai-api-management-gw.azure-api.net",
 api_version="2023-05-15",
 deployment="text-embedding-ada-002",
 api_key=os.environ["AZURE_OPENAI_API_KEY"]
)
#Store them in FAISS vector store for efficient retrieval
vectorstore = FAISS.from_documents(docs, embeddings)

#Initialize the LLM and setup the RAG chain
llm = AzureChatOpenAI(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2025-01-01-preview",
    deployment_name="gpt-5-mini"
    )

#Initialize the RAG pipeline
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

#Query the RAG agent and view results
query = "What is the return policy?"
result= qa_chain.invoke({"query": query})

#Display the answer and the source documents
print("Answer:", result['result'])
print("\n--------Source Documents:-----------")
for i, doc in enumerate(result['source_documents']):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)